# Streaming: live Wikipedia changes relevant to the IMDb dataset

This notebook runs a **PySpark Structured Streaming** pipeline that ingests live Wikipedia change events through **Kafka** and keeps those that concern a film or a person of the IMDb dataset.

```
Wikimedia EventStreams (SSE)  →  wiki_producer.py  →  Kafka topic wikipedia.recentchange  →  Spark Structured Streaming  →  Parquet + metrics
   all wikis, ~50-100 events/s     raw JSON, key = wiki      3 partitions, 24 h retention         parse, filter, join, window
```

**Before running it:**

1. Run `IMDb_Data_Quality_and_Exploration.ipynb`, and ideally `IMDb_Data_Enrichment.ipynb`, with the same `MAX_ROWS` (a full run, `MAX_ROWS = -1`, gives a much larger lookup table).
2. Start Kafka and the producer: `docker compose --profile streaming up -d`.

## Filtering strategy

An event is relevant when it is a content edit or page creation (`type` in `edit`, `new`; main namespace) on English Wikipedia or Wikidata, **and** its page title is in a lookup table built with Spark from the IMDb data:

| Match rule | Page titles in the lookup | Precision |
|---|---|---|
| `enrichment_article` | The exact English Wikipedia articles of the enriched films and principals (`IMDb_Data_Enrichment.ipynb`) | Exact (matched through Wikidata's IMDb ID) |
| `enrichment_wikidata` | The Wikidata items (`Q…`) of the enriched films and principals, for edits on `wikidatawiki` | Exact |
| `imdb_film_title` | `Title`, `Title (film)` and `Title (YEAR film)` for every movie with at least `MIN_FILM_VOTES` IMDb votes | High for the suffixed forms, lower for short common titles |
| `imdb_series_title` | `Title` and `Title (TV series)` for every TV series and mini-series with at least `MIN_FILM_VOTES` votes | High for the suffixed form |
| `imdb_person_name` | The names of the actors, directors, writers, producers and composers of those movies and series | Good, but common names can match a namesake |

When the same page title has several rules, the most precise one is kept. The filter is deliberately broad: English Wikipedia alone receives a few edits per second, and pages about well-known films and people are edited constantly, so a 15–30 minute window contains many matches. The `match_rule` column lets the precision of each rule be assessed afterwards.

## Late-arriving data

- **Event time, not arrival time.** Windows use the time of the edit (`timestamp` field of the event), not the time Spark reads it. The Kafka timestamp is also kept, and `lateness_seconds` = Kafka time − event time measures how late each event is.
- **Watermark of `WATERMARK` (10 minutes).** Spark tracks the latest event time seen. An event up to 10 minutes older than that is still counted in its correct 5-minute window. A window's result is final, and written to the "finalized windows" output, only once the watermark has passed its end. Older events are dropped from the windowed counts, and Spark reports how many in `numRowsDroppedByWatermark`.
- **Why 10 minutes.** EventStreams normally delivers events within seconds (see the lateness measured below). Late events mostly come from a producer reconnection, when the stream resumes from `Last-Event-ID` and replays a few minutes of history. 10 minutes covers such gaps while keeping state small and results reasonably fresh.
- **Duplicates.** A resumed stream can deliver an event twice. `dropDuplicatesWithinWatermark` on the event id removes them.
- **What happens to events later than the watermark.** They are dropped from every live output (matched events and windows) and counted in the `rows_dropped_by_watermark` metric, so the loss is measured, not silent. The raw events stay in Kafka for 24 hours, so they can still be reprocessed in batch from the topic if a complete recount is needed.
- **Trade-off.** A longer watermark accepts later data but keeps more state and delays final results; a shorter one gives faster final windows but drops more late events. The measured lateness below is used to check that 10 minutes is enough.

## 1. Setup

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession, functions as F, types as T

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

SPARK_DATA_DIR = Path(os.environ.get("SPARK_DATA_DIR", "spark_data"))
MAX_ROWS = int(os.environ.get("MAX_ROWS", "3000"))
RUN_NAME = "full" if MAX_ROWS < 0 else f"sample_{MAX_ROWS}"
WORK_DIR = SPARK_DATA_DIR / RUN_NAME
OUTPUT_DIR = Path("results") / RUN_NAME / "streaming"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP", "kafka:9092")
KAFKA_TOPIC = os.environ.get("KAFKA_TOPIC", "wikipedia.recentchange")
STARTING_OFFSETS = "latest"
STREAM_DURATION_MINUTES = float(os.environ.get("STREAM_DURATION_MINUTES", "20"))
TRIGGER_INTERVAL = "30 seconds"
WINDOW = "5 minutes"
WATERMARK = "10 minutes"
MIN_FILM_VOTES = 1000
WIKIS = ["enwiki", "wikidatawiki"]

STREAM_RUN_ID = time.strftime("%Y%m%d_%H%M%S")
STREAM_DIR = WORK_DIR / "streaming" / STREAM_RUN_ID

spark = (
    SparkSession.builder
    .appName("imdb-wikipedia-streaming")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3")
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "3g"))
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Run {STREAM_RUN_ID}: lookup from {WORK_DIR}, Kafka {KAFKA_BOOTSTRAP}/{KAFKA_TOPIC}, "
      f"{STREAM_DURATION_MINUTES:g} minutes, results in {OUTPUT_DIR}/")


def export(df, filename, n=20):
    pdf = df.toPandas()
    display(pdf.head(n))
    pdf.to_csv(OUTPUT_DIR / filename, index=False)
    return pdf

Run 20261115_200237: lookup from /home/jovyan/spark-data/sample_3000, Kafka kafka:9092/wikipedia.recentchange, 5 minutes, results in results/sample_3000/streaming/


## 2. Build the lookup of relevant page titles (Spark batch)

The cleaned IMDb tables come from `IMDb_Data_Quality_and_Exploration.ipynb`; the exact article titles and Wikidata items come from `IMDb_Data_Enrichment.ipynb`, when it has been run.

In [2]:
for table in ["titles_clean", "ratings_clean", "principals_clean", "people_clean"]:
    path = WORK_DIR / "clean" / table
    if not (path / "_SUCCESS").exists():
        raise FileNotFoundError(f"{path} is missing. Run IMDb_Data_Quality_and_Exploration.ipynb first with MAX_ROWS = {MAX_ROWS}.")
    spark.read.parquet(str(path)).createOrReplaceTempView(table)

ENRICHMENT_TABLES = ["wikidata_films_clean", "wikidata_people_clean", "wikipedia_films_clean", "wikipedia_people_clean"]
has_enrichment = all((WORK_DIR / "enrichment" / t / "_SUCCESS").exists() for t in ENRICHMENT_TABLES)
if has_enrichment:
    for table in ENRICHMENT_TABLES:
        spark.read.parquet(str(WORK_DIR / "enrichment" / table)).createOrReplaceTempView(table)
    enrichment_sql = '''
    UNION ALL
    SELECT 'enwiki', p.resolved_title, w.imdb_id, 'film', 'enrichment_article', 1
    FROM wikidata_films_clean w JOIN wikipedia_films_clean p ON p.wikipedia_url = w.wikipedia_url
    WHERE p.resolved_title IS NOT NULL
    UNION ALL
    SELECT 'enwiki', p.resolved_title, w.imdb_id, 'person', 'enrichment_article', 1
    FROM wikidata_people_clean w JOIN wikipedia_people_clean p ON p.wikipedia_url = w.wikipedia_url
    WHERE p.resolved_title IS NOT NULL
    UNION ALL
    SELECT 'wikidatawiki', wikidata_id, imdb_id, 'film', 'enrichment_wikidata', 1 FROM wikidata_films_clean WHERE wikidata_id IS NOT NULL
    UNION ALL
    SELECT 'wikidatawiki', wikidata_id, imdb_id, 'person', 'enrichment_wikidata', 1 FROM wikidata_people_clean WHERE wikidata_id IS NOT NULL
    '''
else:
    enrichment_sql = ""
    print("Enrichment tables not found: only IMDb title and name matching is used.")

spark.sql(f'''
WITH films AS (
    SELECT t.tconst, t.primaryTitle, t.startYear
    FROM titles_clean t JOIN ratings_clean r ON r.tconst = t.tconst
    WHERE t.titleType = 'movie' AND r.numVotes >= {MIN_FILM_VOTES} AND t.primaryTitle IS NOT NULL
),
series AS (
    SELECT t.tconst, t.primaryTitle
    FROM titles_clean t JOIN ratings_clean r ON r.tconst = t.tconst
    WHERE t.titleType IN ('tvSeries', 'tvMiniSeries') AND r.numVotes >= {MIN_FILM_VOTES} AND t.primaryTitle IS NOT NULL
),
works AS (SELECT tconst FROM films UNION SELECT tconst FROM series),
people AS (
    SELECT DISTINCT p.nconst, n.primaryName
    FROM principals_clean p
    JOIN works f ON f.tconst = p.tconst
    JOIN people_clean n ON n.nconst = p.nconst
    WHERE p.category IN ('actor', 'actress', 'director', 'writer', 'producer', 'composer')
      AND n.primaryName IS NOT NULL
),
candidates AS (
    SELECT 'enwiki' AS wiki, primaryTitle AS page_title, tconst AS imdb_id, 'film' AS entity_type,
           'imdb_film_title' AS match_rule, 3 AS priority FROM films
    UNION ALL
    SELECT 'enwiki', concat(primaryTitle, ' (film)'), tconst, 'film', 'imdb_film_title', 2 FROM films
    UNION ALL
    SELECT 'enwiki', concat(primaryTitle, ' (', startYear, ' film)'), tconst, 'film', 'imdb_film_title', 2
    FROM films WHERE startYear IS NOT NULL
    UNION ALL
    SELECT 'enwiki', primaryTitle, tconst, 'series', 'imdb_series_title', 3 FROM series
    UNION ALL
    SELECT 'enwiki', concat(primaryTitle, ' (TV series)'), tconst, 'series', 'imdb_series_title', 2 FROM series
    UNION ALL
    SELECT 'enwiki', primaryName, nconst, 'person', 'imdb_person_name', 3 FROM people
    {enrichment_sql}
),
ranked AS (
    SELECT *, MIN(priority) OVER (PARTITION BY wiki, page_title) AS best_priority FROM candidates
)
SELECT wiki, page_title,
       FIRST(entity_type) AS entity_type,
       FIRST(match_rule) AS match_rule,
       slice(array_sort(collect_set(imdb_id)), 1, 10) AS imdb_ids
FROM ranked
WHERE priority = best_priority
GROUP BY wiki, page_title
''').write.mode("overwrite").parquet(str(STREAM_DIR / "lookup"))

lookup = spark.read.parquet(str(STREAM_DIR / "lookup")).cache()
lookup_summary = lookup.groupBy("wiki", "entity_type", "match_rule").count().orderBy("wiki", "match_rule").toPandas()  # Size of the lookup of relevant page titles, by matching rule
display(lookup_summary)

,wiki,entity_type,match_rule,count
0,enwiki,film,enrichment_article,2090
1,enwiki,person,enrichment_article,4678
2,enwiki,film,imdb_film_title,5794
3,enwiki,person,imdb_person_name,16695
4,enwiki,series,imdb_series_title,670
5,wikidatawiki,film,enrichment_wikidata,2090
6,wikidatawiki,person,enrichment_wikidata,5073


## 3. Define the streaming pipeline


In [3]:
EVENT_SCHEMA = T.StructType([
    T.StructField("meta", T.StructType([
        T.StructField("id", T.StringType()), T.StructField("dt", T.StringType()),
        T.StructField("domain", T.StringType()), T.StructField("stream", T.StringType()),
        T.StructField("uri", T.StringType()),
    ])),
    T.StructField("id", T.LongType()),
    T.StructField("type", T.StringType()),
    T.StructField("namespace", T.IntegerType()),
    T.StructField("title", T.StringType()),
    T.StructField("title_url", T.StringType()),
    T.StructField("comment", T.StringType()),
    T.StructField("timestamp", T.LongType()),
    T.StructField("user", T.StringType()),
    T.StructField("bot", T.BooleanType()),
    T.StructField("minor", T.BooleanType()),
    T.StructField("server_name", T.StringType()),
    T.StructField("wiki", T.StringType()),
    T.StructField("length", T.StructType([T.StructField("old", T.LongType()), T.StructField("new", T.LongType())])),
    T.StructField("revision", T.StructType([T.StructField("old", T.LongType()), T.StructField("new", T.LongType())])),
])

kafka_stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", STARTING_OFFSETS)
    .option("maxOffsetsPerTrigger", 50000)
    .option("failOnDataLoss", "false")
    .load()
)

events = (
    kafka_stream
    .select(F.col("timestamp").alias("kafka_time"),
            F.from_json(F.col("value").cast("string"), EVENT_SCHEMA).alias("e"))
    .select("kafka_time", "e.*")
    .where(F.col("meta.id").isNotNull() & F.col("timestamp").isNotNull())
    .withColumn("event_id", F.col("meta.id"))
    .withColumn("event_time", F.timestamp_seconds("timestamp"))
    .withColumn("lateness_seconds", F.col("kafka_time").cast("double") - F.col("timestamp").cast("double"))
    .withWatermark("event_time", WATERMARK)
    .dropDuplicatesWithinWatermark(["event_id"])
)

matched = (
    events
    .where(F.col("wiki").isin(WIKIS) & (F.col("namespace") == 0) & F.col("type").isin("edit", "new"))
    .join(F.broadcast(lookup.withColumnRenamed("wiki", "lookup_wiki")),
          (F.col("wiki") == F.col("lookup_wiki")) & (F.col("title") == F.col("page_title")))
    .select(
        "event_id", "event_time", "kafka_time", "lateness_seconds", "wiki", "title", "title_url",
        "type", "user", "bot", "minor", "comment",
        (F.col("length.new") - F.col("length.old")).alias("bytes_change"),
        F.col("revision.new").alias("revision_id"),
        "entity_type", "match_rule", "imdb_ids",
        F.current_timestamp().alias("processed_time"),
    )
)

window_counts = (
    matched
    .groupBy(F.window("event_time", WINDOW).alias("window"), "entity_type")
    .agg(F.count("*").alias("matched_events"),
         F.approx_count_distinct("title").alias("distinct_pages"),
         F.sum(F.col("bot").cast("int")).alias("bot_events"),
         F.sum("bytes_change").alias("bytes_change"))
)

all_counts = (
    events
    .withColumn("wiki_group", F.when(F.col("wiki").isin(WIKIS), F.col("wiki")).otherwise(F.lit("other wikis")))
    .groupBy(F.window("event_time", WINDOW).alias("window"), "wiki_group")
    .agg(F.count("*").alias("events"))
)

## 4. Run the stream

In [4]:
queries = {
    "matched_events": matched.writeStream.queryName("matched_events").format("parquet")
        .option("path", str(STREAM_DIR / "matched_events"))
        .option("checkpointLocation", str(STREAM_DIR / "checkpoints" / "matched_events"))
        .outputMode("append").trigger(processingTime=TRIGGER_INTERVAL).start(),
    "final_windows": window_counts.writeStream.queryName("final_windows").format("parquet")
        .option("path", str(STREAM_DIR / "final_windows"))
        .option("checkpointLocation", str(STREAM_DIR / "checkpoints" / "final_windows"))
        .outputMode("append").trigger(processingTime=TRIGGER_INTERVAL).start(),
    "live_windows": window_counts.writeStream.queryName("live_windows").format("memory")
        .outputMode("complete").trigger(processingTime=TRIGGER_INTERVAL).start(),
    "all_windows": all_counts.writeStream.queryName("all_windows").format("memory")
        .outputMode("complete").trigger(processingTime=TRIGGER_INTERVAL).start(),
}

progress_rows, seen_batches = [], set()
end_time = time.time() + STREAM_DURATION_MINUTES * 60
while time.time() < end_time:
    time.sleep(30)
    for name, query in queries.items():
        if query.exception():
            raise query.exception()
        for progress in query.recentProgress:
            if (name, progress["batchId"]) in seen_batches:
                continue
            seen_batches.add((name, progress["batchId"]))
            state = progress.get("stateOperators") or [{}]
            progress_rows.append({
                "query": name,
                "batch_id": progress["batchId"],
                "trigger_time": progress["timestamp"],
                "input_rows": progress.get("numInputRows"),
                "input_rows_per_second": progress.get("inputRowsPerSecond"),
                "processed_rows_per_second": progress.get("processedRowsPerSecond"),
                "batch_duration_ms": progress.get("durationMs", {}).get("triggerExecution"),
                "watermark": progress.get("eventTime", {}).get("watermark"),
                "state_rows": sum(s.get("numRowsTotal", 0) for s in state),
                "rows_dropped_by_watermark": sum(s.get("numRowsDroppedByWatermark", 0) for s in state),
                "sink_rows": progress.get("sink", {}).get("numOutputRows"),
            })
    matched_so_far = spark.read.parquet(str(STREAM_DIR / "matched_events")).count() \
        if (STREAM_DIR / "matched_events" / "_spark_metadata").exists() else 0
    remaining = max(0, int(end_time - time.time()))
    print(f"{time.strftime('%H:%M:%S')}  matched events so far: {matched_so_far:,}  ({remaining // 60} min {remaining % 60} s left)")

for query in queries.values():
    query.stop()
print("Stream stopped.")

20:07:14  matched events so far: 1  (4 min 29 s left)


20:07:45  matched events so far: 1  (3 min 59 s left)


20:08:15  matched events so far: 1  (3 min 29 s left)


20:08:45  matched events so far: 1  (2 min 59 s left)


20:09:15  matched events so far: 1  (2 min 29 s left)


20:09:45  matched events so far: 2  (1 min 58 s left)


20:10:16  matched events so far: 4  (1 min 28 s left)


20:10:46  matched events so far: 5  (0 min 58 s left)


20:11:16  matched events so far: 6  (0 min 27 s left)


20:11:46  matched events so far: 9  (0 min 0 s left)
Stream stopped.


## 5. Results

In [5]:
matched_events = spark.read.parquet(str(STREAM_DIR / "matched_events")).cache()
matched_events_pdf = export(  # Live Wikipedia/Wikidata change events relevant to the IMDb dataset
    matched_events.orderBy("event_time").select(
        "event_time", "wiki", "title", "entity_type", "match_rule", "imdb_ids", "type", "user", "bot",
        "bytes_change", "comment", "lateness_seconds", "title_url"),
    "matched_events.csv",
)

match_summary = export(  # Number of relevant events by entity type and matching rule
    matched_events.groupBy("entity_type", "match_rule").agg(
        F.count("*").alias("events"), F.countDistinct("title").alias("distinct_pages"),
        F.countDistinct("user").alias("distinct_editors"), F.sum(F.col("bot").cast("int")).alias("bot_events"))
    .orderBy(F.desc("events")),
    "match_summary.csv",
)

top_pages = export(  # Most edited IMDb-related pages during the test window
    matched_events.groupBy("wiki", "title", "entity_type", "match_rule").agg(
        F.count("*").alias("events"), F.countDistinct("user").alias("editors"),
        F.sum("bytes_change").alias("bytes_change"), F.first("imdb_ids").alias("imdb_ids"))
    .orderBy(F.desc("events")),
    "top_pages.csv",
)

,event_time,wiki,title,entity_type,match_rule,imdb_ids,type,user,bot,bytes_change,comment,lateness_seconds,title_url
0,2026-11-15 20:06:54,enwiki,Meet Joe Black,film,enrichment_article,[tt0119643],edit,~2026-57078-07,False,-266,/* Filming */ so what? has nothing to do with this film,2.397,https://en.wikipedia.org/wiki/Meet_Joe_Black
1,2026-11-15 20:09:20,enwiki,Raffaella De Laurentiis,person,imdb_person_name,[nm0209581],edit,ThaesOfereode,False,-71,Image,2.523,https://en.wikipedia.org/wiki/Raffaella_De_Laurentiis
2,2026-11-15 20:09:29,enwiki,Raffaella De Laurentiis,person,imdb_person_name,[nm0209581],edit,ThaesOfereode,False,-2,"Changing [[Wikipedia:Short description|short description]] from ""Italian film producer"" to ""Italian film producer (b...",2.996,https://en.wikipedia.org/wiki/Raffaella_De_Laurentiis
3,2026-11-15 20:09:55,enwiki,Raffaella De Laurentiis,person,imdb_person_name,[nm0209581],edit,ThaesOfereode,False,-48,Remove stub template(s) with [[User:DannyS712/De-Stub|De-Stub.js]] (version 1.2),2.950,https://en.wikipedia.org/wiki/Raffaella_De_Laurentiis
4,2026-11-15 20:10:09,enwiki,Yes Man,film,imdb_film_title,[tt1068680],edit,Thepharoah17,False,46,Added {{[[:Template:R from move|R from move]]}} tag to redirect,1.538,https://en.wikipedia.org/wiki/Yes_Man
5,2026-11-15 20:10:49,enwiki,Pacific Rim (film),film,enrichment_article,[tt1663662],edit,~2026-59192-17,False,-34,Jaegers are not robots,8.121,https://en.wikipedia.org/wiki/Pacific_Rim_(film)
6,2026-11-15 20:11:01,enwiki,Brad Bird,person,imdb_person_name,[nm0083348],edit,EpicGuyArts,False,0,/* Style and themes */ Fixed typo,7.961,https://en.wikipedia.org/wiki/Brad_Bird
7,2026-11-15 20:11:04,enwiki,Isy Suttie,person,enrichment_article,[nm2981969],edit,Belbury,False,-64,Reverted 1 edit by [[Special:Contributions/~2026-60484-24|~2026-60484-24]] ([[User talk:~2026-60484-24|talk]]): Trivia,3.210,https://en.wikipedia.org/wiki/Isy_Suttie
8,2026-11-15 20:11:09,wikidatawiki,Q18630946,person,enrichment_wikidata,[nm2486357],edit,Lucamauri,False,347,"/* wbcreateclaim-create:1| */ [[Property:P8979]]: Q13220, [[:toollabs:quickstatements/#/batch/295633|batch #295633]]",1.652,https://www.wikidata.org/wiki/Q18630946


,entity_type,match_rule,events,distinct_pages,distinct_editors,bot_events
0,person,imdb_person_name,4,2,2,0
1,film,enrichment_article,2,2,2,0
2,person,enrichment_wikidata,1,1,1,0
3,film,imdb_film_title,1,1,1,0
4,person,enrichment_article,1,1,1,0


,wiki,title,entity_type,match_rule,events,editors,bytes_change,imdb_ids
0,enwiki,Raffaella De Laurentiis,person,imdb_person_name,3,1,-121,[nm0209581]
1,enwiki,Brad Bird,person,imdb_person_name,1,1,0,[nm0083348]
2,enwiki,Isy Suttie,person,enrichment_article,1,1,-64,[nm2981969]
3,enwiki,Meet Joe Black,film,enrichment_article,1,1,-266,[tt0119643]
4,enwiki,Pacific Rim (film),film,enrichment_article,1,1,-34,[tt1663662]
5,enwiki,Yes Man,film,imdb_film_title,1,1,46,[tt1068680]
6,wikidatawiki,Q18630946,person,enrichment_wikidata,1,1,347,[nm2486357]


### Windowed counts and filter selectivity

In [6]:
final_windows_path = STREAM_DIR / "final_windows"
final_windows = export(  # Final 5-minute windows (written once the watermark passed them)
    spark.read.parquet(str(final_windows_path)).orderBy("window", "entity_type")
    if (final_windows_path / "_spark_metadata").exists()
    else spark.createDataFrame([], "window struct<start:timestamp,end:timestamp>, entity_type string, matched_events long"),
    "final_windows.csv",
)

selectivity = export(  # Relevant events compared with all events received, per 5-minute window
    spark.sql('''
    SELECT a.window.start AS window_start, a.window.end AS window_end,
           SUM(a.events) AS all_events,
           SUM(CASE WHEN a.wiki_group IN ('enwiki', 'wikidatawiki') THEN a.events ELSE 0 END) AS enwiki_wikidata_events,
           COALESCE(MAX(m.matched_events), 0) AS matched_events
    FROM all_windows a
    LEFT JOIN (SELECT window, SUM(matched_events) AS matched_events FROM live_windows GROUP BY window) m
           ON m.window = a.window
    GROUP BY a.window
    ORDER BY window_start
    '''),
    "selectivity_by_window.csv",
)

,window,entity_type,matched_events,distinct_pages,bot_events,bytes_change


,window_start,window_end,all_events,enwiki_wikidata_events,matched_events
0,2026-11-15 20:05:00,2026-11-15 20:10:00,7950,2518,4
1,2026-11-15 20:10:00,2026-11-15 20:15:00,3399,1309,5


### Late data and streaming metrics

In [7]:
lateness = export(  # How late events arrived: Kafka time minus event time, in seconds
    matched_events.select(
        F.count("*").alias("events"),
        F.round(F.percentile_approx("lateness_seconds", 0.5), 2).alias("median_s"),
        F.round(F.percentile_approx("lateness_seconds", 0.95), 2).alias("p95_s"),
        F.round(F.percentile_approx("lateness_seconds", 0.99), 2).alias("p99_s"),
        F.round(F.max("lateness_seconds"), 2).alias("max_s"),
        F.sum((F.col("lateness_seconds") > 600).cast("int")).alias("later_than_watermark"),
    ),
    "lateness.csv",
)

streaming_metrics = pd.DataFrame(progress_rows)  # Per-batch streaming metrics (throughput, latency, watermark, dropped late rows)
display(streaming_metrics.tail(20))
streaming_metrics.to_csv(OUTPUT_DIR / "streaming_metrics.csv", index=False)

metrics_summary = (  # Summary of the streaming metrics per query
    streaming_metrics.groupby("query").agg(
        batches=("batch_id", "nunique"),
        input_rows=("input_rows", "sum"),
        mean_input_rows_per_second=("input_rows_per_second", "mean"),
        mean_processed_rows_per_second=("processed_rows_per_second", "mean"),
        mean_batch_duration_ms=("batch_duration_ms", "mean"),
        max_batch_duration_ms=("batch_duration_ms", "max"),
        rows_dropped_by_watermark=("rows_dropped_by_watermark", "max"),
    ).round(1)
    if not streaming_metrics.empty else pd.DataFrame()
)
display(metrics_summary)
metrics_summary.to_csv(OUTPUT_DIR / "streaming_metrics_summary.csv")

,events,median_s,p95_s,p99_s,max_s,later_than_watermark
0,9,2.95,8.12,8.12,8.12,0


,query,batch_id,trigger_time,input_rows,input_rows_per_second,processed_rows_per_second,batch_duration_ms,watermark,state_rows,rows_dropped_by_watermark,sink_rows
24,matched_events,6,2026-11-15T20:09:30.000Z,1210,40.333333,365.338164,3312,2026-11-15T19:58:58.000Z,6636,0,-1
25,final_windows,6,2026-11-15T20:09:30.000Z,1210,40.333333,297.078321,4073,2026-11-15T19:58:58.000Z,6638,0,-1
26,live_windows,6,2026-11-15T20:09:30.000Z,1210,40.333333,293.404462,4124,2026-11-15T19:58:58.000Z,6638,0,2
27,all_windows,6,2026-11-15T20:09:30.001Z,1210,40.331989,398.944939,3033,2026-11-15T19:58:58.000Z,6639,0,3
28,matched_events,7,2026-11-15T20:10:00.000Z,1249,41.633333,327.220330,3817,2026-11-15T19:59:28.000Z,7885,0,-1
29,final_windows,7,2026-11-15T20:10:00.000Z,1249,41.633333,352.327221,3545,2026-11-15T19:59:28.000Z,7887,0,-1
30,live_windows,7,2026-11-15T20:10:00.000Z,1249,41.633333,300.096108,4162,2026-11-15T19:59:28.000Z,7887,0,2
31,all_windows,7,2026-11-15T20:10:00.000Z,1249,41.634721,465.697241,2682,2026-11-15T19:59:28.000Z,7888,0,3
32,matched_events,8,2026-11-15T20:10:30.000Z,1176,39.200000,533.575318,2204,2026-11-15T19:59:58.000Z,9061,0,-1
33,final_windows,8,2026-11-15T20:10:30.000Z,1176,39.200000,504.504505,2331,2026-11-15T19:59:58.000Z,9064,0,-1


,batches,input_rows,mean_input_rows_per_second,mean_processed_rows_per_second,mean_batch_duration_ms,max_batch_duration_ms,rows_dropped_by_watermark
query,,,,,,,
all_windows,11,11349,36.0,378.1,3883.1,12295,0
final_windows,11,11349,35.9,279.0,4519.9,10107,0
live_windows,11,11349,36.0,252.0,4902.3,10858,0
matched_events,11,11349,35.9,354.1,4031.5,11919,0
